In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install scikit-learn matplotlib pandas tqdm

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder
import warnings


warnings.filterwarnings("ignore", category=UserWarning)


ROOT_DIR = "/content/drive/MyDrive/t-SNE & Probing"
categorical_features = ["gender", "l1_background"]


model_tags = [
    "whisper-tiny.en", "whisper-tiny",
    "whisper-base.en", "whisper-base",
    "whisper-small.en", "whisper-small",
    "whisper-medium.en", "whisper-medium",
    "whisper-large", "whisper-large-v2",
    "whisper-large-v3", "whisper-large-v3-turbo",

    "parakeet-tdt-0.6b-v2",
    "canary-1b-flash",
    "canary-1b",
    "granite-speech-3.3-2b",
    "canary-qwen-2.5b",
    "Phi-4-multimodal-instruct",

    "hubert-large-ls960-ft",
    "hubert-xlarge-ls960-ft",
    "wav2vec2-large-960h-lv60",
    "wavlm-large",

    "speechbrain-loq",
    "w2v2-conformer",


]


dataset_keys = [
    "cam_assess", "SAA", "l2-arctic-dataset-250", "sandi",
    "CommonVoice_accent_stratified", "cmu-arctic-train", "ALLSSTAR_2"
]




def make_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)




def plot_tsne_from_pickle(pkl_path, model_tag, dataset_key):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)


    reps_by_layer = data["reps_by_layer"]
    labels = data["labels"]


    save_dir = os.path.join(ROOT_DIR, "TSNE", model_tag, dataset_key)
    make_folder(save_dir)


    encoders = {}
    for feat in categorical_features:
        if feat in labels and all(v is not None for v in labels[feat]):
            encoder = LabelEncoder()
            labels[feat] = encoder.fit_transform(labels[feat])
            encoders[feat] = encoder


    for feat, y in labels.items():
        if y is None or len(y) == 0 or any(v is None for v in y):
            continue


        save_path = os.path.join(save_dir, f"{model_tag}_{dataset_key}_{feat}_tsne.png")
        if os.path.exists(save_path):
            print(f"✅ Already exists: {save_path}, skipping.")
            continue


        is_categorical = feat in categorical_features
        n_layers = len(reps_by_layer)
        ncols = 4
        nrows = int(np.ceil(n_layers / ncols))
        fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4 * ncols, 4 * nrows))
        axs = axs.flatten()


        sc = None
        for i, layer in enumerate(reps_by_layer):
            tsne_2d = TSNE(n_components=2, random_state=42).fit_transform(layer)
            cmap = "tab10" if is_categorical else "viridis"
            sc = axs[i].scatter(tsne_2d[:, 0], tsne_2d[:, 1], c=y, cmap=cmap, s=10, edgecolor="k", linewidth=0.2)
            axs[i].set_title(f"Layer {i}")
            axs[i].set_xlabel("t-SNE 1")
            axs[i].set_ylabel("t-SNE 2")
            axs[i].grid(True)


        for j in range(i + 1, len(axs)):
            axs[j].axis("off")


        fig.subplots_adjust(right=0.85, wspace=0.4, hspace=0.6)
        if is_categorical and feat in encoders:
            handles, _ = sc.legend_elements()
            labels_ = encoders[feat].inverse_transform(range(len(handles)))
            fig.legend(handles, labels_, title=feat, loc='center left', bbox_to_anchor=(0.88, 0.5))
        elif not is_categorical:
            cbar_ax = fig.add_axes([0.87, 0.15, 0.02, 0.7])
            cbar = fig.colorbar(sc, cax=cbar_ax)
            cbar.set_label(feat)


        plt.savefig(save_path, dpi=300)
        plt.close()
        print(f"✅ Saved t-SNE: {save_path}")




# === MAIN LOOP ===
for model_tag in model_tags:
    for dataset_key in dataset_keys:
        pkl_path = f"/content/drive/MyDrive/Layer Representations/{model_tag}/{dataset_key}.pkl"
        if not os.path.exists(pkl_path):
            print(f"❌ Missing .pkl: {pkl_path}")
            continue


        plot_tsne_from_pickle(pkl_path, model_tag, dataset_key)


print("\n🎯 Done with t-SNE visualizations.")

for fig 1:

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

ROOT_DIR = "/content/drive/MyDrive/t-SNE & Probing"
target_feature = "l1_background"  # ✅ Only run for this feature

model_tags = [
    "whisper-medium.en"
]

dataset_keys = [
    "l2-arctic-dataset-250"
]

def make_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

def plot_tsne_from_pickle(pkl_path, model_tag, dataset_key):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    reps_by_layer = data["reps_by_layer"]
    labels = data["labels"]

    # ✅ Only process if target feature exists
    if target_feature not in labels:
        print(f"⚠️ '{target_feature}' not found in labels for {dataset_key}. Skipping.")
        return

    y = labels[target_feature]
    if y is None or len(y) == 0 or any(v is None for v in y):
        print(f"⚠️ Invalid or empty labels for '{target_feature}'. Skipping.")
        return

    save_dir = os.path.join(ROOT_DIR, "TSNE", model_tag, dataset_key)
    make_folder(save_dir)

    encoder = LabelEncoder()
    y = encoder.fit_transform(y)

    save_path = os.path.join(save_dir, f"{model_tag}_{dataset_key}_{target_feature}_tsne.png")
    if os.path.exists(save_path):
        print(f"✅ Already exists: {save_path}, skipping.")
        return

    n_layers = len(reps_by_layer)
    ncols = 4
    nrows = int(np.ceil(n_layers / ncols))
    fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4 * ncols, 4 * nrows))
    axs = axs.flatten()

    sc = None
    for i, layer in enumerate(reps_by_layer):
        tsne_2d = TSNE(n_components=2, random_state=42).fit_transform(layer)
        sc = axs[i].scatter(tsne_2d[:, 0], tsne_2d[:, 1], c=y, cmap="tab10", s=10, edgecolor="k", linewidth=0.2)
        axs[i].set_title(f"Layer {i}")
        axs[i].set_xlabel("t-SNE 1")
        axs[i].set_ylabel("t-SNE 2")
        axs[i].grid(True)

    for j in range(i + 1, len(axs)):
        axs[j].axis("off")

    fig.subplots_adjust(right=0.85, wspace=0.4, hspace=0.6)
    handles, _ = sc.legend_elements()
    labels_ = encoder.inverse_transform(range(len(handles)))
    fig.legend(handles, labels_, title=target_feature, loc='center left', bbox_to_anchor=(0.88, 0.5))

    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ Saved t-SNE: {save_path}")

# === MAIN LOOP ===
for model_tag in model_tags:
    for dataset_key in dataset_keys:
        pkl_path = f"/content/drive/MyDrive/Layer Representations/{model_tag}/{dataset_key}.pkl"
        if not os.path.exists(pkl_path):
            print(f"❌ Missing .pkl: {pkl_path}")
            continue

        plot_tsne_from_pickle(pkl_path, model_tag, dataset_key)

print("\n🎯 Done with t-SNE visualizations (only l1_background).")
